# UGV Standalone — Visual Navigation Analysis

This notebook analyses the front-camera RGB-D datasets without YOLO. Complete simulation runs remain intact in the train, validation and test sets, preventing adjacent frames from the same trajectory leaking between sets.

Nav2 commands are teacher labels. The notebook compares the current visual rules with constant, linear-ridge and quadratic-ridge imitation baselines. These are analysis models only; they never publish commands to the Husky.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_root():
    for start in (Path.cwd(), Path.cwd().parent):
        for root in (start, *start.parents):
            if (root/'UGV_Standalone').is_dir() and (root/'datasets').is_dir():
                return root
    raise FileNotFoundError('Run from the EiraX workspace or a child folder.')

ROOT=find_root()
DATASETS=ROOT/'datasets'
print('Workspace:', ROOT)


In [ ]:
def mission_complete(run_dir):
    events=run_dir/'events.jsonl'
    return events.exists() and '"event":"mission_completed"' in events.read_text(errors='replace')

runs=[]
for labels in sorted(DATASETS.glob('run_*/visual_navigation/labels.csv')):
    run_dir=labels.parents[1]
    frame=pd.read_csv(labels)
    frame['run_id']=run_dir.name
    frame['sample_index']=np.arange(len(frame))
    teacher_ready={'nav2_linear_mps','nav2_angular_rps'}.issubset(frame.columns)
    runs.append({'run_id':run_dir.name,'path':run_dir,'complete':mission_complete(run_dir),'teacher_ready':teacher_ready,'data':frame})

catalog=pd.DataFrame([{'run_id':r['run_id'],'samples':len(r['data']),'complete':r['complete'],'teacher_ready':r['teacher_ready']} for r in runs])
display(catalog)
assert len(catalog[catalog.complete & catalog.teacher_ready])>=2, 'At least two completed teacher-labelled runs are required.'


## Leakage-safe run split

Incomplete and manually positioned runs are shown in the catalogue but excluded from training. Runs recorded before Nav2 teacher-command columns were introduced are also excluded. With three or more eligible missions, complete missions are assigned chronologically at approximately 60/20/20%. If only two are available, the notebook uses one for training and preserves the newest as untouched test data; validation is explicitly marked pending rather than leaking trajectory frames.

In [ ]:
eligible=[r for r in runs if r['complete'] and r['teacher_ready']]
n=len(eligible)
if n>=3:
    n_train=max(1,int(np.floor(.60*n)))
    n_val=max(1,int(np.floor(.20*n)))
    if n_train+n_val>=n: n_train,n_val=n-2,1
    split_runs={'train':eligible[:n_train],'validation':eligible[n_train:n_train+n_val],'test':eligible[n_train+n_val:]}
else:
    split_runs={'train':[eligible[0]],'validation':[],'test':[eligible[-1]]}
    print('VALIDATION PENDING: collect one more completed teacher-labelled mission. Fixed hyperparameters will be used; the test run remains untouched.')
splits={name:(pd.concat([r['data'] for r in group],ignore_index=True) if group else eligible[0]['data'].iloc[0:0].copy())
        for name,group in split_runs.items()}
for name,group in split_runs.items():
    print(f"{name:10s}: {[r['run_id'] for r in group]} ({len(splits[name])} samples)")


In [ ]:
FEATURES=['target_x','confidence','obstacle_distance_m','odom_yaw']
TARGETS=['nav2_linear_mps','nav2_angular_rps']

def clean(df):
    out=df.copy()
    out['obstacle_distance_m']=(pd.to_numeric(out['obstacle_distance_m'],errors='coerce')
        .replace([np.inf,-np.inf],np.nan).fillna(15.0).clip(0,15))
    required=FEATURES+TARGETS+['safe_linear_mps','safe_angular_rps']
    for col in required:
        out[col]=pd.to_numeric(out[col],errors='coerce')
    return out.dropna(subset=required).reset_index(drop=True)

splits={k:clean(v) for k,v in splits.items()}
train,validation,test=splits['train'],splits['validation'],splits['test']
display(train[FEATURES].describe().round(3))


In [ ]:
mu=train[FEATURES].to_numpy().mean(axis=0)
sigma=train[FEATURES].to_numpy().std(axis=0)
sigma[sigma<1e-8]=1.0

def design(df,quadratic=False):
    z=(df[FEATURES].to_numpy()-mu)/sigma
    columns=[np.ones((len(z),1)),z]
    if quadratic:
        columns += [z*z, np.column_stack([z[:,i]*z[:,j]
                    for i in range(z.shape[1]) for j in range(i+1,z.shape[1])])]
    return np.column_stack(columns)

def fit_ridge(df,alpha,quadratic=False):
    X,y=design(df,quadratic),df[TARGETS].to_numpy()
    penalty=np.eye(X.shape[1]); penalty[0,0]=0
    return np.linalg.solve(X.T@X+alpha*penalty,X.T@y)

def safety_filter(pred,df):
    p=pred.copy()
    p[:,0]=np.clip(p[:,0],0,.65)
    p[:,1]=np.clip(p[:,1],-.55,.55)
    confidence=df.confidence.to_numpy()
    obstacle=df.obstacle_distance_m.to_numpy()
    p[confidence<.25]=0
    p[obstacle<=1.5]=0
    p[:,0]*=np.clip((obstacle-1.5)/2.0,0,1)
    return p

def metrics(pred,df):
    truth=df[TARGETS].to_numpy()
    turn=lambda a:np.where(a>.04,1,np.where(a<-.04,-1,0))
    nav_stop=truth[:,0]<.02
    pred_stop=pred[:,0]<.02
    return {'linear_MAE':np.mean(abs(pred[:,0]-truth[:,0])),
            'angular_MAE':np.mean(abs(pred[:,1]-truth[:,1])),
            'turn_agreement':np.mean(turn(pred[:,1])==turn(truth[:,1])),
            'teacher_stop_recall':np.mean(pred_stop[nav_stop]) if nav_stop.any() else np.nan,
            'predicted_stop_rate':np.mean(pred_stop)}


In [ ]:
if len(validation):
    candidates=[]
    constant=np.tile(train[TARGETS].mean().to_numpy(),(len(validation),1))
    candidates.append(('constant',None,False,safety_filter(constant,validation)))
    for quadratic in (False,True):
        for alpha in (.01,.1,1.,10.,100.):
            weights=fit_ridge(train,alpha,quadratic)
            pred=safety_filter(design(validation,quadratic)@weights,validation)
            candidates.append(('quadratic_ridge' if quadratic else 'linear_ridge',alpha,quadratic,pred))
    validation_table=pd.DataFrame([{'model':name,'alpha':alpha,**metrics(pred,validation)} for name,alpha,_,pred in candidates])
    validation_table['selection_score']=validation_table.linear_MAE+validation_table.angular_MAE+.25*(1-validation_table.teacher_stop_recall.fillna(1))
    display(validation_table.sort_values('selection_score').round(4))
else:
    validation_table=pd.DataFrame([{'model':'linear_ridge','alpha':1.0,'selection_score':np.nan}])
    print('No validation metrics reported: linear ridge alpha=1.0 was fixed before examining the held-out test run.')


In [ ]:
best=validation_table.sort_values('selection_score').iloc[0]
quadratic=best.model=='quadratic_ridge'
combined=pd.concat([train,validation],ignore_index=True) if len(validation) else train.copy()
if best.model=='constant':
    learned=np.tile(combined[TARGETS].mean().to_numpy(),(len(test),1))
else:
    weights=fit_ridge(combined,float(best.alpha),quadratic)
    learned=design(test,quadratic)@weights
learned=safety_filter(learned,test)
visual=test[['safe_linear_mps','safe_angular_rps']].to_numpy()
constant_test=safety_filter(
    np.tile(combined[TARGETS].mean().to_numpy(),(len(test),1)),test)
results=pd.DataFrame([
    {'model':'current_RGB-D_rules',**metrics(visual,test)},
    {'model':'constant_baseline',**metrics(constant_test,test)},
    {'model':f"selected_{best.model}",**metrics(learned,test)}])
display(results.round(4))
print('Selected on validation:',best.model,'alpha=',best.alpha)


In [ ]:
truth=test[TARGETS].to_numpy()
x=np.arange(len(test))
fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=True)
for values,label in [(truth,'Nav2 teacher'),(visual,'Current RGB-D rules'),
                     (learned,'Selected learned model')]:
    axes[0].plot(x,values[:,0],label=label,lw=1)
    axes[1].plot(x,values[:,1],label=label,lw=1)
axes[0].set_ylabel('linear m/s'); axes[0].legend()
axes[1].set_ylabel('angular rad/s'); axes[1].set_xlabel('held-out test sample')
plt.tight_layout(); plt.show()


## Latest verified result (2026-08-24)

Independent run split: training `run_20260824_203353`, validation `run_20260824_212511`, test `run_20260824_214714`. The selected quadratic ridge model (alpha 100) achieved linear/angular MAE of 0.1953/0.1041 and 59.33% turn agreement, but recalled only 15% of Nav2 stop samples. The current RGB-D rules achieved 100% stop recall with linear/angular MAE of 0.2592/0.1259 and 56.58% turn agreement. Therefore the learned model is **not approved for vehicle control**; the deterministic RGB-D and LiDAR safety path remains the valid implementation.

## Deployment gate

Lower imitation error does not prove safe autonomy. Keep LiDAR emergency stopping, stale-sensor stopping, speed limits and the no-reverse constraint outside any learned policy.

Do not enable direct visual control until at least five completed missions cover open ground, buildings, trees, narrow passages and lighting variation, followed by a successful guarded waypoint-1 test. Deep reinforcement learning should be trained in simulation only after this supervised baseline is reproducible; it must not learn collision safety from scratch.